# Modeling 

In [1]:
# import both datasets with significant features and target value 

from pathlib import Path # access to working directory 
import pandas as pd # data manipulation 

# current working directory 
dir = Path.cwd()

# look into Data and find linear.csv and non_linear.csv
data_path = dir.parent / 'Data' / 'linear.csv'
data_path2 = dir.parent / 'Data' / 'non_linear.csv'
target_path = dir.parent / 'Data' / 'target.csv'

# load data path into csv
linear = pd.read_csv(data_path)
non_linear = pd.read_csv(data_path2)
target = pd.read_csv(target_path)


In [2]:
# training phase for simpler models
# import sklearn train test split 
from sklearn.model_selection import train_test_split

# target value
Y = target['Survived']

# covariate variables (linear)
X = linear[['Pclass', 'Age', 'SibSp', 'Sex_female', 'Cabin_encoded']]

# 60/40 train/test split 
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size= 0.3, shuffle= True, random_state = 42)


In [3]:
# Modeling Phase (Classification)
# import sklearn numeric performance indicators for downstream modeling workflow 
from sklearn.metrics import confusion_matrix, classification_report 


## logistic Regression
Performance Metrics: 
- Recall: 81% 
- Precision: 75% 
- Fl Score: 78% 

Results: 
- The model did fairly well on accuarately predicting the survived or died classes. In terms of the recall metric, I do believe that 75% is decent when compared with a high harmonic mean. The harmonic mean in this case isn't above the well respected threshold to determine if the model has any use case of being effective in the real world. Also, model performance dropped slightly after hyper tunning and fitting parameters which suggest a possible overfit. 

In [4]:
# basline model (logistic regression) 

from sklearn.linear_model import LogisticRegression

# begin instantiantion process
lr = LogisticRegression(max_iter= 100)

# fit data on model
lr.fit(X_train, Y_train)

# predictions 
lr_pred = lr.predict(X_test)

# performance metrrics
print(f'Classification Report {classification_report(Y_test, lr_pred)}')
print(f'Confusion matric {confusion_matrix(Y_test, lr_pred)}')


Classification Report               precision    recall  f1-score   support

           0       0.84      0.87      0.86       157
           1       0.81      0.77      0.79       111

    accuracy                           0.83       268
   macro avg       0.83      0.82      0.82       268
weighted avg       0.83      0.83      0.83       268

Confusion matric [[137  20]
 [ 26  85]]


In [5]:
# Let's find the best hyperparameter settings for the model 
from sklearn.model_selection import GridSearchCV

# parameters
params = [ 
    {
        'solver': ['liblinear'], 
        'penalty': ['l1', 'l2'], 
        'C': [0.001, 0.01, 0.1, 1, 10, 50, 100],
        'fit_intercept': [True], 
    
    }, 
    { 
        'solver': ['lbfgs'], 
        'penalty': ['l2'], # lbfgs cannot do L1
        'C': [0.001, 0.1, 1, 10, 100],
        'fit_intercept': [True]

    }, 
    {
        'solver': ['saga'], 
        'penalty': ['elasticnet'],
        'C': [0.01, 0.1, 1, 10, 100], 
        'l1_ratio': [0.5], # required for elasticnet 
        'fit_intercept': [True]
    }
]


# instantiantion process 
lr_best = GridSearchCV(estimator= lr, param_grid= params, cv = 10, verbose= 5, n_jobs= -1, scoring = 'recall')

# fit data on model
lr_best.fit(X_train, Y_train)

# predictions
lr_pred = lr_best.predict(X_test)

# performance metrics 
print(f'Classification Report C{classification_report(Y_test, lr_pred)}')
print(f'Confusion Matrix {confusion_matrix(Y_test, lr_pred)}')
print(f'Best settings {lr_best.best_estimator_}')

Fitting 10 folds for each of 24 candidates, totalling 240 fits
Classification Report C              precision    recall  f1-score   support

           0       0.83      0.88      0.85       157
           1       0.81      0.75      0.78       111

    accuracy                           0.82       268
   macro avg       0.82      0.81      0.82       268
weighted avg       0.82      0.82      0.82       268

Confusion Matrix [[138  19]
 [ 28  83]]
Best settings LogisticRegression(C=10, penalty='l1', solver='liblinear')


## Support Vector Machines

Peformance Metrics:
- Recall: 80% 
- precision: 76% 
- F1: 78%  

Key Insights: 
- In the process of utilizing GridSearchCv, the model had trouble converging. High numerical SVM parameters work extremely hard to achieve 100% accuracy on the training data. Thus, giving the model a hard time to come up with a prediction within a certiain timeframe even with the implementation of n_jobs (a parameters that tells the model to use all cpu's inside of a computer). The optimal solution in this case was to envoke HavlingGridSearchCV (a derivitive of GridSearchCV with the ability to use a subset of data points, leading to faster predictions and best setting outputs)

Results:
- The Model only caught 80% of survivors and accurately predicted 76% These numeric values indicate a possible overfit of the model seeing as the best parameters included a moderate C. The F1 metric also signals a slight usage in the real world. 

In [6]:
# training phase (non linearity)
from sklearn.model_selection import train_test_split 

# target variable 
Y = target['Survived']

# inputs 
X = non_linear.copy()

# 70/30 split 
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.3, random_state = 42) 


In [7]:
from sklearn.svm import SVC # support vector model 
# instantiation process
svm = SVC()

# fit the data on the model
svm.fit(X_train, Y_train)

# predictios
Y_predictions = svm.predict(X_test)

# predictions results
Y_predictions

# Evaluate model performance results 
print(f'Classification Report {classification_report(Y_test, Y_predictions)}')
print(f'Confusion Matrix {confusion_matrix(Y_test, Y_predictions)}')

Classification Report               precision    recall  f1-score   support

           0       0.75      0.92      0.83       157
           1       0.84      0.57      0.68       111

    accuracy                           0.78       268
   macro avg       0.80      0.75      0.75       268
weighted avg       0.79      0.78      0.77       268

Confusion Matrix [[145  12]
 [ 48  63]]


In [14]:
# find the best settings for the SVM model 
from sklearn.experimental import enable_halving_search_cv # required!
from sklearn.model_selection import HalvingGridSearchCV


# parameters
params = [
    {
        'kernel': ['linear'], 
        'C': [0.1, 1, 10, 100]
        
    },

    {
        'kernel': ['rbf'], 
        'C': [0.1, 1, 10, 100], 
        'gamma': [0.0001, 0.001, 0.1, 1]
     
    },
    {
        'kernel': ['poly'], 
        'degree': [2, 3, 4], 
        'C': [0.1, 1, 10, 100] 
    
    }
]
# Grid Search settings 
svm_best = HalvingGridSearchCV(estimator = svm, param_grid = params, factor = 6,  resource = 'n_samples', n_jobs = -1, verbose = 5, scoring = 'recall', cv= 10)
# fit data onto model
svm_best.fit(X_train, Y_train)

# predictions
svm_pred = svm_best.predict(X_test)

# performance result 
print(f'Classification Report: {classification_report(Y_test, svm_pred)}')
print(f'confusion Matrix: {confusion_matrix(Y_test, svm_pred)}')
print(f'Best Settings: {svm_best.best_estimator_}')

n_iterations: 2
n_required_iterations: 2
n_possible_iterations: 2
min_resources_: 103
max_resources_: 623
aggressive_elimination: False
factor: 6
----------
iter: 0
n_candidates: 32
n_resources: 103
Fitting 10 folds for each of 32 candidates, totalling 320 fits
----------
iter: 1
n_candidates: 6
n_resources: 618
Fitting 10 folds for each of 6 candidates, totalling 60 fits
Classification Report:               precision    recall  f1-score   support

           0       0.85      0.82      0.84       157
           1       0.76      0.80      0.78       111

    accuracy                           0.81       268
   macro avg       0.81      0.81      0.81       268
weighted avg       0.82      0.81      0.81       268

confusion Matrix: [[129  28]
 [ 22  89]]
Best Settings: SVC(C=10, gamma=1)


## K-NearestNeighbors Algorithm

Metrics: 
- Recall: 84% 
- Precision: 70% 
- F1 Score: 76%

Results: After tuning the model to optimize for recall, the performance dipped slightly from the baseline. This is a sign of overfitting! Moving foward dowstream we will be sticking to the baseline KNN if it makes it into the production phase. 



In [15]:
# (KNN) uses distances to classify which class a data point belongs in 

from sklearn.neighbors import KNeighborsClassifier 

# instantiation process
KNN = KNeighborsClassifier(n_neighbors=5)

# fit the data on the model 
KNN.fit(X_train, Y_train)

# predictions
KNN_pred = KNN.predict(X_test)

# performance results 
print(f'Classification Report: {classification_report(Y_test, KNN_pred)}')
print(f'Confusion Matric: {confusion_matrix(Y_test, KNN_pred)}')

Classification Report:               precision    recall  f1-score   support

           0       0.81      0.90      0.86       157
           1       0.84      0.70      0.76       111

    accuracy                           0.82       268
   macro avg       0.83      0.80      0.81       268
weighted avg       0.82      0.82      0.82       268

Confusion Matric: [[142  15]
 [ 33  78]]


In [16]:
# Find the best optimal settings for K-Nearest Neighbors 
# import gridsearchcv to locate the best paramaters for the model
from sklearn.model_selection import GridSearchCV

params = {'n_neighbors': [3, 5, 7, 8, 11, 15, 21], 
          'weights': ['uniform', 'distance'], 
          'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'], 
          'metric': ['euclidean', 'manhattan']}

# instantiation process
KNN_best = GridSearchCV(estimator = KNN, param_grid= params, cv = 10, verbose = 5, scoring = 'recall')

# fit data on the model 
KNN_best.fit(X_train, Y_train)

# predictions 
KNN_pred = KNN_best.predict(X_test)

# performance results
print(f'Classification Report:{classification_report(Y_test, KNN_pred)}')
print(f'Best Settings: {(KNN_best.best_estimator_)}')

Fitting 10 folds for each of 112 candidates, totalling 1120 fits
[CV 1/10] END algorithm=auto, metric=euclidean, n_neighbors=3, weights=uniform;, score=0.708 total time=   0.0s
[CV 2/10] END algorithm=auto, metric=euclidean, n_neighbors=3, weights=uniform;, score=0.609 total time=   0.0s
[CV 3/10] END algorithm=auto, metric=euclidean, n_neighbors=3, weights=uniform;, score=0.783 total time=   0.0s
[CV 4/10] END algorithm=auto, metric=euclidean, n_neighbors=3, weights=uniform;, score=0.783 total time=   0.0s
[CV 5/10] END algorithm=auto, metric=euclidean, n_neighbors=3, weights=uniform;, score=0.609 total time=   0.0s
[CV 6/10] END algorithm=auto, metric=euclidean, n_neighbors=3, weights=uniform;, score=0.565 total time=   0.0s
[CV 7/10] END algorithm=auto, metric=euclidean, n_neighbors=3, weights=uniform;, score=0.609 total time=   0.0s
[CV 8/10] END algorithm=auto, metric=euclidean, n_neighbors=3, weights=uniform;, score=0.739 total time=   0.0s
[CV 9/10] END algorithm=auto, metric=eu

## Random Forest Classification
- Metrics: 
- Recall: 71% 
- Precision 79% 
- F1 Score: 73% 

Important Insight: 
- The model struggled to converege under the 4min window, due to the complexity of the params. The solution was to use a the halving model of grid search to chop the time it took to outputt results in half. 

Results: 
- The hypertuned model increased slighly in performance, with a ossibility the model could've performed better if CPU usuage increased. The numerical outtputted scores we're decent but not significant enough to move the model into a production phase. 

In [20]:
# Random Forest Classification 
from sklearn.ensemble import RandomForestClassifier 

# instantiation process
rf = RandomForestClassifier(n_estimators= 400)
# fit data onto model 
rf.fit(X_train, Y_train)

# predictions
rf_pred = rf.predict(X_test)

# peformance evaluation
print(f'Classification Repor: {classification_report(Y_test, rf_pred)}')
print(f'Confusion Matrix: {confusion_matrix(Y_test, rf_pred)}')

Classification Repor:               precision    recall  f1-score   support

           0       0.80      0.83      0.82       157
           1       0.75      0.71      0.73       111

    accuracy                           0.78       268
   macro avg       0.78      0.77      0.78       268
weighted avg       0.78      0.78      0.78       268

Confusion Matrix: [[131  26]
 [ 32  79]]


In [27]:
# Find best parameters for the model 
# params 
params = {'n_estimators': [300, 400, 500, 800], 
          'max_depth': [None, 10, 20, 30], 
          'min_samples_split': [2, 5, 10], 
          'max_features': ['sqrt', 'log2'],
          'random_state': [42], 
          'n_jobs': [-1]}

# instantation process 
rf_best = HalvingGridSearchCV(estimator= rf, param_grid= params, cv = 10, verbose = 5, n_jobs = -1, factor = 3, scoring = 'recall')
# fit data onto model 
rf_best.fit(X_train, Y_train)

# predicitions
rf_pred = rf_best.predict(X_test)

# performance summary 
print(f'Classification Report: {classification_report(Y_test, rf_pred)}')
print(f'Confusion Matrix: {confusion_matrix(Y_test, rf_pred)}')
print(f'Best Settings: {rf_best.best_estimator_}')

n_iterations: 3
n_required_iterations: 5
n_possible_iterations: 3
min_resources_: 40
max_resources_: 623
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 96
n_resources: 40
Fitting 10 folds for each of 96 candidates, totalling 960 fits


C:\Users\dmiracju\AppData\Roaming\Python\Python313\site-packages\numpy\ma\core.py:2896: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


----------
iter: 1
n_candidates: 32
n_resources: 120
Fitting 10 folds for each of 32 candidates, totalling 320 fits
----------
iter: 2
n_candidates: 11
n_resources: 360
Fitting 10 folds for each of 11 candidates, totalling 110 fits


C:\Users\dmiracju\AppData\Roaming\Python\Python313\site-packages\numpy\ma\core.py:2896: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Classification Report:               precision    recall  f1-score   support

           0       0.81      0.87      0.84       157
           1       0.79      0.71      0.75       111

    accuracy                           0.80       268
   macro avg       0.80      0.79      0.79       268
weighted avg       0.80      0.80      0.80       268

Confusion Matrix: [[136  21]
 [ 32  79]]
Best Settings: RandomForestClassifier(min_samples_split=5, n_estimators=300, n_jobs=-1,
                       random_state=42)


# XGboost Algorithm 
Performance Metrics: 
- Recall: 69% 
- Precision: 75% 
- F1: 72%   

Results: Model performed way worse than expected. I'm finding trouble trying to realize if its underfitting or overfitting the data, but in terms of the data structure and training splits the model under performed. 

In [ ]:
# Let's use a boosted algorithm to capture omore accuacry and performance 

# import package 
import xgboost as xgb 

# intialize xgboostclassifier
xgb = xgb.XGBClassifier(n_estimators = 100)
# fit data on boosted algo
xgb.fit(X_train, Y_train)

# predictions
xgb_pred = xgb.predict(X_test)

# performance metrics 
print(f'Classification Report {classification_report(Y_test, xgb_pred)}')
print(f'Confusin Matrix {confusion_matrix(Y_test, xgb_pred)}')


Classification Report               precision    recall  f1-score   support

           0       0.80      0.84      0.82       157
           1       0.75      0.69      0.72       111

    accuracy                           0.78       268
   macro avg       0.78      0.77      0.77       268
weighted avg       0.78      0.78      0.78       268

Confusin Matrix [[132  25]
 [ 34  77]]


In [36]:
# Let's find the best settings for the boosted algorithm

# parameters 
params = {'n_estimators': [100, 300, 500],
          'max_depth': [3, 5, 7, 9], 
          'min_features' 
          'subsample': [0.6, 0.8, 1.0], 
          'colsample_bytree': [0.6, 0.8, 1.0], 
          'gamma': [0, 0.1, 0.2]
}

# intitalize gridsearch 
xgb_best = GridSearchCV(estimator= xgb, param_grid= params, cv = 10, verbose = 5, n_jobs =-1) 

# fit data on model
xgb_best.fit(X_train, Y_train)

# predcitions
xgb_pred = xgb.predict(X_test)

# evaluation metrics 
print(f'Classification Report {classification_report(Y_test, xgb_pred)}')
print(f'Confusion Matirx {confusion_matrix(Y_test, xgb_pred)}')
print(f'Best settings {xgb_best.best_estimator_}')


Fitting 10 folds for each of 324 candidates, totalling 3240 fits
Classification Report               precision    recall  f1-score   support

           0       0.80      0.84      0.82       157
           1       0.75      0.69      0.72       111

    accuracy                           0.78       268
   macro avg       0.78      0.77      0.77       268
weighted avg       0.78      0.78      0.78       268

Confusion Matirx [[132  25]
 [ 34  77]]
Best settings XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=0.1, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step

C:\Users\dmiracju\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:199: UserWarning: [21:16:59] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "min_featuressubsample" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
